[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/40_mlops/02_docker_and_ci_cd.ipynb)

# Bài 02 — Docker, kiểm thử ML và CI/CD

**🎯 Mục tiêu**
- Tổ chức một dự án ML **triển khai được**: package Python (dữ liệu, đặc trưng, huấn luyện, API), script huấn luyện có cổng chất lượng, bộ test.
- Viết **bộ kiểm thử cho ML**: unit test đặc trưng, kiểm tra dữ liệu, kiểm thử hành vi mô hình (hiệu năng tối thiểu, bất biến, kỳ vọng theo hướng), kiểm thử API và parity.
- Đóng gói bằng **Docker**: layer và cache, ghim phiên bản, user không phải root, healthcheck.
- **CI/CD** với GitHub Actions: lint → huấn luyện → test → build image → deploy; khái niệm Continuous Training và các mức trưởng thành MLOps.

Cần `pip install -e ".[mlops]"` và pytest (`dev`). Docker là **tuỳ chọn**: phần build image chỉ chạy khi có Docker và đặt biến môi trường `ML_COURSE_RUN_DOCKER=1`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/40_mlops

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import yaml

from ml_course.datasets import make_customer_churn

PROJECT = Path("artifacts/churn_project")                              # thư mục đã nằm trong .gitignore
for sub in ["churn", "tests", "data", ".github/workflows"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)
make_customer_churn(8000, seed=0).to_csv(PROJECT / "data" / "churn.csv", index=False)   # vai trò "kho dữ liệu"

## 1. Cấu trúc dự án

Notebook tốt cho khám phá; hệ thống chạy từ **package + script + test**:

```
churn_project/
├── churn/                 # package: mọi code dùng chung giữa huấn luyện và phục vụ
│   ├── data.py            #   đọc, kiểm tra, chia dữ liệu
│   ├── features.py        #   đặc trưng tuỳ biến (import được → pickle an toàn với tiến trình khác)
│   ├── train.py           #   huấn luyện + cổng chất lượng + đóng gói mô hình
│   └── api.py             #   dịch vụ FastAPI
├── tests/                 # unit · dữ liệu · mô hình · API
├── model/                 # sản phẩm huấn luyện (không commit — lấy từ registry/CI artifact)
├── requirements.txt       # phụ thuộc runtime, ghim phiên bản
├── pyproject.toml         # cấu hình pytest
├── Dockerfile, .dockerignore
└── .github/workflows/ci.yml
```

Nguyên tắc: **một nguồn sự thật** cho đặc trưng — `churn/features.py` được dùng cả khi train lẫn khi phục vụ (chống lệch train/serving, chương 39).

In [ ]:
%%writefile artifacts/churn_project/churn/__init__.py
__version__ = "1.1.0"

In [ ]:
%%writefile artifacts/churn_project/churn/data.py
"""Đọc, kiểm tra và chia dữ liệu huấn luyện."""
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
TARGET = "churn"
CATEGORIES = {"contract": {"month-to-month", "one_year", "two_year"}, "internet_service": {"dsl", "fiber", "none"},
              "payment_method": {"bank_transfer", "cash", "credit_card", "e-wallet"}}


def load_data(path) -> pd.DataFrame:
    return pd.read_csv(path)


def validate_training_data(df: pd.DataFrame) -> list[str]:
    """Danh sách vấn đề của dữ liệu huấn luyện; rỗng = hợp lệ."""
    missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
    if missing:
        return [f"thiếu cột {missing}"]
    issues = []
    if len(df) < 1000:
        issues.append(f"quá ít dòng: {len(df)}")
    if df[FEATURES + [TARGET]].isna().any().any():
        issues.append("có giá trị thiếu")
    for col, allowed in CATEGORIES.items():
        unknown = set(df[col].unique()) - allowed
        if unknown:
            issues.append(f"{col}: giá trị lạ {sorted(unknown)}")
    if (df.monthly_charges <= 0).any() or (df.tenure_months < 0).any():
        issues.append("giá trị số ngoài miền hợp lệ")
    for col in ["support_calls", "monthly_charges", "tenure_months"]:
        if df[col].nunique() <= 1:
            issues.append(f"{col} là hằng số — nghi lỗi pipeline dữ liệu")
    if not 0.05 <= df[TARGET].mean() <= 0.6:
        issues.append(f"tỉ lệ churn bất thường: {df[TARGET].mean():.2f}")
    return issues


def split(df: pd.DataFrame, seed: int = 0):
    return train_test_split(df[FEATURES], df[TARGET], test_size=0.25, random_state=seed, stratify=df[TARGET])

In [ ]:
%%writefile artifacts/churn_project/churn/features.py
"""Đặc trưng tuỳ biến — dùng chung cho huấn luyện và phục vụ."""
import pandas as pd


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    expected_total = out.tenure_months.clip(lower=1) * out.monthly_charges
    out["billing_ratio"] = (out.total_charges / expected_total).clip(0, 3)   # ≈ 1 nếu thanh toán đều
    out["is_new_customer"] = (out.tenure_months < 6).astype(int)
    return out

Script huấn luyện là một **chương trình dòng lệnh** (chạy được trong CI, cron, Airflow…), có **cổng chất lượng**: dữ liệu không hợp lệ hoặc AUC dưới ngưỡng → thoát với mã lỗi ≠ 0 → pipeline CI dừng, không có mô hình tệ nào được đóng gói.

In [ ]:
%%writefile artifacts/churn_project/churn/train.py
"""Huấn luyện + đóng gói: python -m churn.train --data data/churn.csv --out model --min-auc 0.78"""
import argparse
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from churn import __version__
from churn.data import load_data, split, validate_training_data
from churn.features import add_features

NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products", "billing_ratio", "is_new_customer"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]


def build_model():
    columns = ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(FunctionTransformer(add_features), columns, LogisticRegression(max_iter=1000))


def main(argv=None) -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--out", default="model")
    parser.add_argument("--min-auc", type=float, default=0.78)
    args = parser.parse_args(argv)

    df = load_data(args.data)
    issues = validate_training_data(df)
    if issues:
        print("❌ dữ liệu không hợp lệ:", issues)
        return 1
    X_tr, X_te, y_tr, y_te = split(df)
    model = build_model().fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    if auc < args.min_auc:
        print(f"❌ AUC holdout {auc:.4f} < ngưỡng {args.min_auc}")
        return 1

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, out / "model.joblib")
    meta = {"name": "churn", "version": __version__, "holdout_roc_auc": auc, "n_train": len(X_tr),
            "data_sha256": hashlib.sha256(Path(args.data).read_bytes()).hexdigest(),
            "model_sha256": hashlib.sha256((out / "model.joblib").read_bytes()).hexdigest(),
            "python": sys.version.split()[0], "sklearn": sklearn.__version__,
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    (out / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"✅ đã đóng gói churn {__version__} — AUC holdout {auc:.4f} → {out}/")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile artifacts/churn_project/churn/api.py
"""Dịch vụ dự đoán churn (rút gọn từ bài 01)."""
import hashlib
import json
import os
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, ConfigDict, Field

MODEL_DIR = Path(os.environ.get("MODEL_DIR", "model"))
STATE: dict = {}


class CustomerFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")
    tenure_months: int = Field(ge=0, le=600)
    contract: Literal["month-to-month", "one_year", "two_year"]
    internet_service: Literal["dsl", "fiber", "none"]
    monthly_charges: float = Field(gt=0, le=1000)
    total_charges: float = Field(ge=0)
    payment_method: Literal["bank_transfer", "cash", "credit_card", "e-wallet"]
    support_calls: int = Field(ge=0, le=100)
    senior: bool
    n_products: int = Field(ge=1, le=20)


class Prediction(BaseModel):
    churn_probability: float
    model_version: str


@asynccontextmanager
async def lifespan(app: FastAPI):
    meta = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))
    if hashlib.sha256((MODEL_DIR / "model.joblib").read_bytes()).hexdigest() != meta["model_sha256"]:
        raise RuntimeError("model.joblib không khớp hash")
    STATE["model"], STATE["meta"] = joblib.load(MODEL_DIR / "model.joblib"), meta
    yield
    STATE.clear()


app = FastAPI(title="Churn API", lifespan=lifespan)


@app.get("/health")
def health():
    return {"status": "ok", "model_version": STATE["meta"]["version"]}


@app.post("/v1/predict", response_model=list[Prediction])
def predict(customers: list[CustomerFeatures]):
    probs = STATE["model"].predict_proba(pd.DataFrame([c.model_dump() for c in customers]))[:, 1]
    return [Prediction(churn_probability=p, model_version=STATE["meta"]["version"]) for p in probs]

## 2. Kiểm thử cho ML

Phần mềm thường kiểm thử **code**; hệ thống ML phải kiểm thử thêm **dữ liệu** và **hành vi mô hình** (Breck et al., 2017 — *ML Test Score*; Ribeiro et al., 2020 — *CheckList*):

| Lớp | Kiểm tra | Ví dụ trong dự án |
|---|---|---|
| Unit | Hàm đặc trưng đúng, xử lý biên, không sửa đầu vào | `billing_ratio` không chia cho 0 khi `tenure = 0` |
| Dữ liệu | Schema, giá trị thiếu, miền giá trị, cột hằng, tỉ lệ nhãn | `validate_training_data` bắt cột `support_calls` toàn 0 |
| Mô hình — hiệu năng tối thiểu | Metric trên holdout cố định ≥ ngưỡng; tái lập metric | AUC ≥ 0.78; bằng đúng số trong metadata |
| Mô hình — bất biến (invariance) | Thay đổi **không** liên quan → dự đoán không đổi | thêm cột `customer_id`, đảo thứ tự cột |
| Mô hình — kỳ vọng theo hướng (directional) | Thay đổi có hướng đã biết → dự đoán đổi đúng hướng | nhiều cuộc gọi hỗ trợ hơn → rủi ro cao hơn |
| API | Hợp đồng, mã lỗi, **parity** với mô hình | 422 khi sai; API khớp `predict_proba` |

In [ ]:
%%writefile artifacts/churn_project/pyproject.toml
[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]
filterwarnings = ["ignore:Using .httpx. with", "ignore::DeprecationWarning:starlette"]

In [ ]:
%%writefile artifacts/churn_project/tests/conftest.py
import json
from pathlib import Path

import joblib
import pytest

from churn.data import load_data, split

ROOT = Path(__file__).resolve().parents[1]


@pytest.fixture(scope="session")
def data():
    return load_data(ROOT / "data" / "churn.csv")


@pytest.fixture(scope="session")
def holdout(data):
    _, X_te, _, y_te = split(data)
    return X_te, y_te


@pytest.fixture(scope="session")
def model():
    return joblib.load(ROOT / "model" / "model.joblib")


@pytest.fixture(scope="session")
def meta():
    return json.loads((ROOT / "model" / "metadata.json").read_text(encoding="utf-8"))

In [ ]:
%%writefile artifacts/churn_project/tests/test_features.py
import pandas as pd

from churn.features import add_features


def make_rows(**overrides):
    row = {"tenure_months": 10, "monthly_charges": 50.0, "total_charges": 500.0}
    return pd.DataFrame([{**row, **overrides}])


def test_billing_ratio_is_one_for_regular_payment():
    assert abs(add_features(make_rows()).billing_ratio.iloc[0] - 1.0) < 1e-12


def test_new_customer_without_division_by_zero():
    out = add_features(make_rows(tenure_months=0, total_charges=0.0))
    assert out.billing_ratio.iloc[0] == 0.0 and out.is_new_customer.iloc[0] == 1


def test_boundary_of_new_customer():
    assert add_features(make_rows(tenure_months=6)).is_new_customer.iloc[0] == 0


def test_does_not_mutate_input():
    df = make_rows()
    add_features(df)
    assert list(df.columns) == ["tenure_months", "monthly_charges", "total_charges"]

In [ ]:
%%writefile artifacts/churn_project/tests/test_data.py
from churn.data import validate_training_data


def test_training_data_is_valid(data):
    assert validate_training_data(data) == []


def test_detects_constant_column(data):
    assert any("support_calls" in issue for issue in validate_training_data(data.assign(support_calls=0)))


def test_detects_unknown_category(data):
    broken = data.copy()
    broken.loc[0, "contract"] = "monthly"
    assert any("contract" in issue for issue in validate_training_data(broken))


def test_detects_missing_values(data):
    broken = data.copy()
    broken.loc[0, "monthly_charges"] = None
    assert "có giá trị thiếu" in validate_training_data(broken)

In [ ]:
%%writefile artifacts/churn_project/tests/test_model.py
import numpy as np
from sklearn.metrics import roc_auc_score


def test_minimum_performance(model, holdout):
    X_te, y_te = holdout
    assert roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]) >= 0.78


def test_metric_is_reproducible(model, holdout, meta):
    X_te, y_te = holdout
    assert np.isclose(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), meta["holdout_roc_auc"])


def test_invariance_to_irrelevant_columns_and_order(model, holdout):
    X_te, _ = holdout
    base = model.predict_proba(X_te)[:, 1]
    shuffled_columns = X_te[X_te.columns[::-1]].assign(customer_id="C99999")
    assert np.allclose(model.predict_proba(shuffled_columns)[:, 1], base)


def test_more_support_calls_means_higher_risk(model, holdout):
    X_te, _ = holdout
    more_calls = X_te.assign(support_calls=X_te.support_calls + 3)
    assert (model.predict_proba(more_calls)[:, 1] > model.predict_proba(X_te)[:, 1]).all()


def test_longer_contract_means_lower_risk(model, holdout):
    X_te, _ = holdout
    monthly = model.predict_proba(X_te.assign(contract="month-to-month"))[:, 1]
    two_year = model.predict_proba(X_te.assign(contract="two_year"))[:, 1]
    assert (two_year < monthly).mean() > 0.95

In [ ]:
%%writefile artifacts/churn_project/tests/test_api.py
import json
import os
from pathlib import Path

import numpy as np
import pytest
from fastapi.testclient import TestClient

os.environ["MODEL_DIR"] = str(Path(__file__).resolve().parents[1] / "model")
from churn.api import app  # noqa: E402


@pytest.fixture(scope="module")
def client():
    with TestClient(app) as c:
        yield c


def test_health(client, meta):
    assert client.get("/health").json() == {"status": "ok", "model_version": meta["version"]}


def test_rejects_invalid_input(client, holdout):
    record = json.loads(holdout[0].iloc[:1].to_json(orient="records"))[0]
    assert client.post("/v1/predict", json=[{**record, "contract": "monthly"}]).status_code == 422
    assert client.post("/v1/predict", json=[{**record, "unknown_field": 1}]).status_code == 422


def test_parity_with_model(client, model, holdout):
    X = holdout[0].iloc[:100]
    response = client.post("/v1/predict", json=json.loads(X.to_json(orient="records")))
    api = np.array([p["churn_probability"] for p in response.json()])
    assert np.allclose(api, model.predict_proba(X)[:, 1])

## 3. Chạy pipeline như CI

Hai bước CI sẽ chạy: **huấn luyện** (có cổng chất lượng) rồi **pytest**. Ta chạy đúng các lệnh đó bằng tiến trình con, từ thư mục dự án.

In [ ]:
def run(cmd, cwd=PROJECT):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, encoding="utf-8", errors="replace",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(f"$ {' '.join(cmd[2:] if cmd[0] == sys.executable else cmd)}  → mã thoát {result.returncode}")
    print("\n".join((result.stdout + result.stderr).strip().splitlines()[-19:]))
    return result.returncode


assert run([sys.executable, "-m", "churn.train", "--data", "data/churn.csv", "--out", "model"]) == 0
assert run([sys.executable, "-m", "pytest", "-v", "--color=no", "-p", "no:cacheprovider"]) == 0

Cổng chất lượng hoạt động: dữ liệu bị lỗi (cột `support_calls` hằng 0 — một join hỏng) hoặc ngưỡng AUC không đạt → script **từ chối** đóng gói, CI đỏ.

In [ ]:
make_customer_churn(8000, seed=0).assign(support_calls=0).to_csv(PROJECT / "data" / "broken.csv", index=False)
run([sys.executable, "-m", "churn.train", "--data", "data/broken.csv", "--out", "model_broken"])
run([sys.executable, "-m", "churn.train", "--data", "data/churn.csv", "--out", "model_strict", "--min-auc", "0.95"])
print("thư mục mô hình được tạo:", sorted(p.name for p in PROJECT.glob("model*")))
(PROJECT / "data" / "broken.csv").unlink()

## 4. Docker

**Image** = hệ điều hành tối giản + Python + thư viện (đúng phiên bản) + code + mô hình, đóng băng thành các **layer**; **container** = một tiến trình chạy từ image. Cùng image chạy giống nhau trên laptop, CI, server — giải quyết "máy tôi chạy được".

Ghim phiên bản phụ thuộc **runtime** bằng đúng phiên bản đã dùng để huấn luyện (mô hình pickle phụ thuộc phiên bản scikit-learn — chương 39):

In [ ]:
runtime = ["numpy", "pandas", "scipy", "scikit-learn", "joblib", "fastapi", "uvicorn", "pydantic"]
requirements = "\n".join(f"{name}=={metadata.version(name)}" for name in runtime) + "\n"
(PROJECT / "requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)

In [ ]:
%%writefile artifacts/churn_project/Dockerfile
# syntax=docker/dockerfile:1
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PIP_NO_CACHE_DIR=1
WORKDIR /app

# (1) Phụ thuộc trước: layer này được cache, chỉ build lại khi requirements.txt đổi
COPY requirements.txt .
RUN pip install -r requirements.txt

# (2) Code và mô hình sau: đổi thường xuyên, build lại nhanh
COPY churn/ churn/
COPY model/ model/

# (3) Không chạy bằng root
RUN useradd --create-home appuser
USER appuser

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health')"
CMD ["uvicorn", "churn.api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]

In [ ]:
%%writefile artifacts/churn_project/.dockerignore
.git
.github
__pycache__/
*.pyc
.pytest_cache/
data/
tests/
model_*/
*.ipynb

| Thực hành | Lý do |
|---|---|
| Image nền `-slim` | nhỏ (≈ 150 MB thay vì ≈ 1 GB), ít lỗ hổng |
| Cùng phiên bản Python với lúc train | pickle/thư viện tương thích |
| `COPY requirements.txt` + `pip install` **trước** `COPY` code | đổi code không phải cài lại thư viện (cache layer) |
| `.dockerignore` loại dữ liệu, test, `.git` | image nhỏ, không lộ dữ liệu |
| `USER appuser` | container bị chiếm quyền cũng không có quyền root |
| `HEALTHCHECK` | orchestrator (Docker, Kubernetes) tự khởi động lại container hỏng |
| Mô hình **trong** image hoặc tải từ registry khi khởi động | trong image: bất biến, dễ rollback; tải ngoài: image nhỏ, đổi mô hình không cần build lại |

Kiểm tra nhất quán đơn giản — phiên bản Python của image phải khớp môi trường huấn luyện:

In [ ]:
base_image = next(line for line in (PROJECT / "Dockerfile").read_text(encoding="utf-8").splitlines() if line.startswith("FROM"))
image_python = base_image.split(":")[1].split("-")[0]
local_python = ".".join(platform.python_version_tuple()[:2])
print(f"{base_image} | Python huấn luyện: {local_python} →", "khớp ✅" if image_python == local_python else "KHÔNG khớp ⚠️ — sửa FROM")

In [ ]:
docker_ok = shutil.which("docker") is not None and os.environ.get("ML_COURSE_RUN_DOCKER") == "1"
commands = [["docker", "build", "-t", "churn-api:1.1.0", "."],
            ["docker", "run", "-d", "--rm", "-p", "8000:8000", "--name", "churn-api", "churn-api:1.1.0"],
            ["docker", "images", "churn-api"]]
if docker_ok:
    for cmd in commands:
        run(cmd)
    print("thử: curl http://127.0.0.1:8000/health — dừng: docker stop churn-api")
else:
    print("Bỏ qua build Docker (đặt ML_COURSE_RUN_DOCKER=1 và bật Docker để chạy). Các lệnh:")
    for cmd in commands:
        print("   ", " ".join(cmd))

## 5. CI/CD với GitHub Actions

**CI** (continuous integration): mỗi push/PR tự động lint → huấn luyện (có cổng) → test. **CD** (continuous delivery/deployment): nhánh `main` qua CI → build image → đẩy lên container registry → deploy staging → (phê duyệt) → production.

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
from matplotlib.patches import FancyBboxPatch

steps = [("push / pull request", "0.8"), ("lint\n(ruff)", "C0"), ("huấn luyện +\ncổng chất lượng", "C1"), ("pytest", "C0"),
         ("build image →\ncontainer registry", "C2"), ("deploy\nstaging", "C2"), ("phê duyệt →\nproduction", "C3")]
fig, ax = plt.subplots(figsize=(14, 3.2))
for i, (label, color) in enumerate(steps):
    ax.add_patch(FancyBboxPatch((i * 2, 0), 1.6, 1, boxstyle="round,pad=0.05", fc=color, alpha=0.35, ec="0.3"))
    ax.text(i * 2 + 0.8, 0.5, label, ha="center", va="center", fontsize=9)
    if i:
        ax.annotate("", xy=(i * 2 - 0.06, 0.5), xytext=(i * 2 - 0.36, 0.5), arrowprops=dict(arrowstyle="-|>", color="0.3"))
ax.annotate("dữ liệu lỗi / metric dưới ngưỡng\n→ CI đỏ, không đóng gói", xy=(4.8, -0.05), xytext=(4.8, -1.0), ha="center",
            fontsize=9, color="#d62728", arrowprops=dict(arrowstyle="-|>", color="#d62728"))
ax.text(3.8, 1.35, "CI — mọi push/PR", ha="center", fontsize=10)
ax.text(10.8, 1.35, "CD — chỉ nhánh main", ha="center", fontsize=10)
ax.plot([0, 7.6], [1.18, 1.18], color="C0", lw=2)
ax.plot([8, 13.6], [1.18, 1.18], color="C2", lw=2)
ax.set_xlim(-0.2, len(steps) * 2 - 0.2)
ax.set_ylim(-1.4, 1.6)
ax.axis("off")
ax.set_title("Pipeline CI/CD: bước nào thất bại thì dừng cả chuỗi")
plt.show()

In [ ]:
%%writefile artifacts/churn_project/.github/workflows/ci.yml
name: churn-ci

on:
  pull_request:
  push:
    branches: [main]
  schedule:
    - cron: "0 2 * * 1"          # Continuous Training: huấn luyện lại mỗi thứ Hai 2h sáng

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt pytest httpx ruff
      - run: ruff check churn tests
      - run: dvc pull data/churn.csv     # dữ liệu không nằm trong git — lấy theo phiên bản (DVC, chương 39)
      - run: python -m churn.train --data data/churn.csv --out model --min-auc 0.78
      - run: pytest -q
      - uses: actions/upload-artifact@v4
        with:
          name: model
          path: model/

  build-and-push:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4
      - uses: actions/download-artifact@v4
        with:
          name: model
          path: model/
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}
      - run: docker build -t ghcr.io/${{ github.repository }}/churn-api:${{ github.sha }} .
      - run: docker push ghcr.io/${{ github.repository }}/churn-api:${{ github.sha }}

  deploy-staging:
    needs: build-and-push
    runs-on: ubuntu-latest
    environment: staging             # environment có thể yêu cầu phê duyệt thủ công
    steps:
      - run: echo "deploy ghcr.io/${{ github.repository }}/churn-api:${{ github.sha }} lên staging, chạy smoke test"

In [ ]:
workflow = yaml.safe_load((PROJECT / ".github" / "workflows" / "ci.yml").read_text(encoding="utf-8"))
for name, job in workflow["jobs"].items():
    steps = [step.get("uses") or step["run"].split(" --")[0][:40] for step in job["steps"]]
    print(f"{name:15s} ← {str(job.get('needs', '—')):15s} | {steps}")
print("kích hoạt bởi:", list(workflow[True]))                          # YAML 1.1 đọc khoá `on` thành True

- Image được gắn tag bằng **commit SHA** → mỗi image truy về đúng code; rollback = deploy lại tag cũ.
- Lịch `schedule` biến pipeline thành **Continuous Training**: huấn luyện lại định kỳ trên dữ liệu mới; cổng chất lượng + cổng thăng hạng (chương 39) quyết định mô hình mới có được dùng không.

**Mức trưởng thành MLOps** (Google Cloud):

| Mức | Đặc điểm | Rủi ro / hạn chế |
|---|---|---|
| 0 — Thủ công | Notebook, huấn luyện tay, copy file mô hình cho kỹ sư | không tái lập, chậm, lệch train/serving |
| 1 — Tự động huấn luyện | Pipeline huấn luyện tự động (CT), kiểm tra dữ liệu/mô hình, registry, giám sát kích hoạt huấn luyện lại | pipeline vẫn deploy tay khi đổi code |
| 2 — CI/CD cho pipeline | Code pipeline cũng qua CI/CD; test, build, deploy pipeline tự động | đầu tư hạ tầng lớn — chỉ cần khi có nhiều mô hình/đội |

Công cụ điều phối pipeline: Airflow, Prefect, Dagster, Kubeflow Pipelines, Vertex AI/SageMaker Pipelines; GitHub Actions đủ cho dự án nhỏ.

## 🧠 Kiểm tra nhanh

1. Vì sao code đặc trưng phải nằm trong package dùng chung cho train và serve?
2. Kể năm loại kiểm thử cho hệ thống ML, mỗi loại một ví dụ.
3. Kiểm thử bất biến và kiểm thử theo hướng khác nhau thế nào?
4. Vì sao `COPY requirements.txt` và `pip install` đặt trước `COPY` code trong Dockerfile?
5. Một pipeline CI cho ML gồm những bước nào? Cổng chất lượng đặt ở đâu?
6. Continuous Training là gì? Khi nào cần MLOps mức 2?

## 📌 Tóm tắt

- Dự án triển khai được = package (dữ liệu, đặc trưng, huấn luyện, API) + script có cổng chất lượng + bộ test + Dockerfile + workflow CI.
- Kiểm thử ML: unit, dữ liệu, hiệu năng tối thiểu và tái lập metric, bất biến, theo hướng, API và parity.
- Docker đóng băng môi trường; ghim phiên bản theo môi trường huấn luyện; tối ưu cache layer; không chạy root; healthcheck.
- CI/CD: lint → train → test → build → push → deploy; tag image bằng commit; lịch chạy định kỳ cho Continuous Training.

## 📚 Tài liệu tham khảo

- Breck et al. (2017) — *The ML Test Score: A Rubric for ML Production Readiness and Technical Debt Reduction*
- Ribeiro et al. (2020) — *Beyond Accuracy: Behavioral Testing of NLP Models with CheckList*
- Google Cloud — *MLOps: Continuous delivery and automation pipelines in machine learning*
- Tài liệu Docker — *Best practices for writing Dockerfiles*; GitHub Docs — *GitHub Actions*
- Kreuzberger et al. (2023) — *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*